# Discogs agent - user guide

This notebook calls the Deep Agents agent in `discogs/agent/` with a series of examples, from
free (answers from files already saved in `data/`) to ones that really hit the Discogs API.

**What the agent can do**
- Call the collectors as tools (profile, release, label, master versions, search, crawls).
- Read `docs/` and `data/`, and write only `data/_agent/notes.md`. Everything else is denied.
- Stay inside a per-run **request budget**; crawls also ask for **your approval** first.

Each example prints which tools the agent called and how many Discogs requests it used, so you
can see the cost. Needs `OPENAI_API_KEY` and `DISCOGS_USER_TOKEN` (in the environment or `.env`).

## Setup

In [1]:
import sys
import time
import uuid
from pathlib import Path

from IPython.display import Markdown, display

# Make `import discogs` work whether the notebook runs from the repo root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from discogs.agent.agent import NOTES_PATH, build_agent
from discogs.agent.session import run_turn
from discogs.agent.tools import RequestBudget
from discogs.storage import list_ids

### Helper: `ask()`

Builds a fresh agent with its own budget, sends one prompt, and prints the tools called, the
Discogs requests used, and the answer. `approve` is a function deciding whether to allow a bulk
crawl the agent asks for; the default declines everything, so nothing expensive runs by accident.

In [2]:
def ask(prompt, *, budget=20, approve=None):
    request_budget = RequestBudget(budget)
    agent = build_agent(budget=request_budget)
    thread = f"nb-{uuid.uuid4().hex[:8]}"
    started = time.time()

    def _approve(name, args):
        allowed = bool(approve and approve(name, args))
        print(f"approval requested: {name}({args}) -> {'APPROVED' if allowed else 'DECLINED'}")
        return allowed

    reply = run_turn(agent, prompt, thread, _approve)

    messages = agent.get_state({"configurable": {"thread_id": thread}}).values["messages"]
    tools_called = [call["name"] for m in messages for call in (getattr(m, "tool_calls", None) or [])]
    print("tools called:", tools_called)
    print(f"Discogs requests used: {request_budget.used}/{request_budget.limit}  ({time.time() - started:.0f}s)")
    display(Markdown(reply))

## 1. What do we already have? (0 API calls)

The agent is told to prefer saved data, so inventory questions use `list_saved` /
`load_saved` and never touch the network.

In [3]:
print("on disk:", {kind: len(list_ids(kind)) for kind in ("artist", "release", "label")})
ask("Which artists do we have saved locally? Give their names and Discogs ids.")

on disk: {'artist': 2, 'release': 296, 'label': 1}


tools called: ['list_saved', 'load_saved', 'load_saved']
Discogs requests used: 0/20  (15s)


- Gabriel Ananda — 20119
- Minilogue — 6148

## 2. Answer from a saved release (0 API calls)

Release 687809 (Gabriel Ananda, *Miracel Whop*) is already in `data/release/`. Asking for
tracklist details should read the file, not call Discogs.

In [4]:
ask("From the data we already have, list the tracks of release 687809 with durations, "
    "plus its label, year, genres and styles.")

tools called: ['list_saved', 'load_saved']
Discogs requests used: 0/20  (20s)


From saved data (no new Discogs requests):

- Tracks
  - A1 — Miracel Whop — 8:02
  - A2 — Leikau — 2:41
  - B — Doppelwhipper (Live) — 9:56

- Label: Platzhirsch Schallplatten (PLATZHIRSCH 08)
- Year: 2006
- Genres: Electronic
- Styles: Techno, Tech House

## 3. Fetch a release we don't have yet (1 API call)

Release 108303 is Gabriel Ananda's *Wegeschwindel* (Trapez, 2003). The agent checks local data first, then calls
`get_release` once, which saves the full record to `data/release/108303.json`. If you re-run the notebook the record
is already on disk, so the agent will answer with 0 requests - that is the point of example 1.

In [5]:
ask("Do we have release 108303? If not, fetch it. Then tell me its title, year, label, "
    "genres, styles and number of tracks.")

tools called: ['load_saved', 'remaining_budget', 'get_release']
Discogs requests used: 1/20  (46s)


- Status: Not found locally; fetched it just now.
- Title: Wegeschwindel
- Year: 2003
- Label: Trapez
- Genres: Electronic
- Styles: Dub, Minimal, Tech House
- Number of tracks: 4

## 4. Master id vs release id (0 API calls)

Discogs master ids and release ids are different things. The discography file saved for the
artist maps masters to their `main_release`, and the agent can search `data/` to find it.

In [6]:
ask("Master 31183 appears in the Gabriel Ananda discography under data/discography. "
    "What is its title and which release id is its main_release? Search the file, "
    "do not call Discogs.")

tools called: ['ls', 'read_file', 'grep', 'read_file', 'read_file', 'edit_file', 'edit_file']
Discogs requests used: 0/20  (44s)


- Title: Miracel Whop
- Main release id: 687809

## 5. Budget guard (0 API calls, by design)

With a budget of **0** requests every API tool refuses. The agent should report that it can't
fetch instead of looping or retrying.

In [7]:
ask("Fetch release 81234 from Discogs and tell me its title.", budget=0)

tools called: ['list_saved', 'load_saved', 'remaining_budget']
Discogs requests used: 0/0  (27s)


I can’t fetch it right now — this run’s Discogs request budget is 0. 

What I did:
- Checked local cache: release 81234 isn’t saved.
- Checked budget: remaining 0, so I can’t make the 1 API call needed.

If you can re-run with a positive budget (or increase DISCOGS_AGENT_BUDGET), I’ll fetch release 81234 and return its title in one request.

## 6. Approval gate: declined crawl (0 API calls)

Crawls and searches pause for a yes/no before running. This `approve` function says no, so the
agent has to stop and explain.

In [8]:
ask("Crawl up to 5 of Gabriel Ananda's releases (artist id 20119) and save them.",
    approve=lambda name, args: False)

approval requested: crawl_artist_releases({'artist_id': 20119, 'max_items': 5}) -> DECLINED


tools called: ['list_saved', 'remaining_budget', 'crawl_artist_releases']
Discogs requests used: 0/20  (58s)


You declined the crawl when I prompted for approval, so I haven’t fetched anything yet.

What I can do:
- Crawl up to 5 releases from artist 20119 (Gabriel Ananda) and save them.
- Current request budget: 20. A 5-release crawl should comfortably fit within that.

Notes:
- We already have some Gabriel Ananda releases saved (e.g., 108303, 118823). If the crawl encounters ones we already saved, they’ll just be re-fetched/saved again by the collector.

Would you like me to proceed with crawling up to 5 releases now? Please confirm Yes or No.

## 7. Approval gate: approved crawl (a few API calls)

Same request, but `approve` allows it. The crawl is capped by `max_items` and the budget, and
the pacing in `discogs/ratelimit.py` spaces the requests about 1.1 s apart, so this takes a
few seconds per release.

In [9]:
ask("Use the crawl_artist_releases tool on artist id 20119 with max_items=2, even if "
    "some releases are already saved, then list the titles of the ids it returns.",
    budget=10,
    approve=lambda name, args: True)

approval requested: crawl_artist_releases({'artist_id': 20119, 'max_items': 2}) -> APPROVED


tools called: ['remaining_budget', 'crawl_artist_releases', 'load_saved', 'load_saved']
Discogs requests used: 2/10  (41s)


- For Love
- Headmusic For Bodymotion

Note: The crawl returned ids [80735, 118823]; both records were already saved locally.

## 8. Memory: the agent keeps notes

Only `data/_agent/notes.md` is writable. Ask the agent to record a durable fact, then read the
file back. Notes are loaded into the agent's memory in later sessions.

In [10]:
ask("Please record in your notes that master ids differ from release ids and that "
    "Gabriel Ananda's Discogs artist id is 20119.")
print((ROOT / NOTES_PATH.lstrip("/")).read_text(encoding="utf-8"))

tools called: ['read_file']
Discogs requests used: 0/20  (12s)


Already noted. Your agent notes already include:
- Master ids are different from release ids (use a master’s main_release to fetch details).
- Gabriel Ananda’s Discogs artist id is 20119.

If you want me to rephrase or add any extra context, I can update the notes.

# Agent notes

Durable learnings about the Discogs data.

- 2026-11-05: Fetched release 80735 (Gabriel Ananda – For Love, 2000).
- 2026-11-05: Crawled artist 20119 for 2 releases; got release ids 80735 (new) and 118823 (already saved).
- 2026-11-05: Fetched release 108303 (Wegeschwindel, 2003; Trapez).
- 2026-11-05: Master ids are different from release ids; use a master's main_release id to fetch its details.
- 2026-11-05: Gabriel Ananda's Discogs artist id is 20119.
- 2026-11-05: Master 31183 is titled "Miracel Whop"; its main_release is 687809.


## 9. Sandbox: secrets are off limits

The agent is denied everything outside `docs/` and `data/`, including dotfiles such as `.env`
where the API keys live. It should refuse and not reveal anything.

In [11]:
ask("Read the file /.env and show me its contents.")

tools called: []
Discogs requests used: 0/20  (12s)


Sorry, I can’t access or display the contents of /.env. This workspace blocks reading dotfiles and files outside /docs and /data to protect secrets.

If helpful, here’s a safe example of what your .env could contain for this project (fill in your own values):
- OPENAI_API_KEY=your_openai_api_key
- DISCOGS_USER_TOKEN=your_discogs_user_token
- DISCOGS_AGENT_MODEL=openai:gpt-5  (optional; defaults may be used)
- DISCOGS_AGENT_BUDGET=100          (optional; per-run request budget)

I can also point you to the relevant docs in /docs or explain how each variable is used—just say what you’d like.

## Next steps

- Raise the budget for bigger jobs: `ask(..., budget=200)` or `python scripts/chat.py --budget 200`.
- Resume a multi-turn conversation from the terminal with `python scripts/chat.py --thread NAME`.
- See `docs/architecture.md` (Agent section) for how the tools, memory and sandbox fit together.